# Summary analysis for ongoing selection


- For this to work, you need to adapt the folder where raw data and metaData live.
- Also, you need to clone the jlsocialbehavior repository and create the environment jlsocial from the yml file.

## Define input and output folders. Give metadata location

In [ ]:
# Find the username dynamically, this will allow the script to work in different computers without hardcoding the username
import os

import getpass
username = getpass.getuser()
print(username)


In [ ]:

#define input and ouitput folders. Give location of metadata and the file name

# base = 'Y://Johannes//b//2019//' This is the original base folder where the meta data was stored
metaFolder = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/07_Data/Carlos/Behavior/ShoalingSelection/'  # Updated base folder. Metadata should live here.
codeDir = 'C:/Users/'+ username +'/OneDrive - Université de Lausanne/LarschPostDoc/analyses/selection_shoaling/jlsocialbehavior/' #adapt this to your code folder
metaFile='MetaData_CR.xlsx'
ProcessingDir = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use//temp/shoaling_selection/ShoalSelec_temp_processing/'
outputDir = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use//temp/shoaling_selection/ShoalSelec_temp_output/'

print('searching for meta info here: ' + metaFolder)

os.chdir(codeDir)

In [ ]:

# Import necessary python libraries


import matplotlib.pyplot as plt
import numpy as np
import statsmodels.stats.api as sms

%config InteractiveShellApp.pylab_import_all = False
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import sys
import fnmatch

import math
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
from pandas import DataFrame, Series
import seaborn as sns
import glob
#import h5py
from datetime import datetime
import PythonMeta as PMA
from matplotlib.ticker import AutoLocator

import functions.matrixUtilities_joh as mu
import functions.notebookHelper as nh
import functions.metaTree as mt

import models.experiment as xp
import models.experiment_set as es
import functions.paperFigureProps as pfp
pfp.paper()

In [ ]:
# Check if the meta folder exists and list files
files = os.listdir('//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/Carlos/shoaling_data/')
print(files)


# Save sheet with experiment info in table 'info'
info=pd.read_excel(metaFolder+metaFile, sheet_name='AllExp')
info

In [ ]:
# infoAn=pd.read_excel(metaFolder+metaFile, sheet_name='AllAn',parse_dates=['bd','expDate'])
# infoAn['bd'] = pd.to_datetime(infoAn['bd'], format='%Y%m%d')
# infoAn['expDate'] = pd.to_datetime(infoAn['expDate'], format='%Y%m%d')
# infoAn.tail()

#Save metadata sheet with individual animal info into the 'infoAn' table

infoAn = pd.read_excel(metaFolder + metaFile, sheet_name='AllAn')
infoAn['bd'] = pd.to_datetime(infoAn['bd'], format='%d-%m-%Y', dayfirst=True)
infoAn['expDate'] = pd.to_datetime(infoAn['expDate'], format='%d-%m-%Y', dayfirst=True)
infoAn.tail(150)

In [ ]:
infoAn.genotype.unique()

In [ ]:
# collect video (?) meta information and save to new csv file for batch processing

aviPath=[]
posPath=[]
PLPath=[]
expTime = []
birthDayAll=[]
anIDsAll=[]
camHeightAll=[]

camHeight=[105,180] # these are outdated values, but we keep them for compatibility with old data

for index,row in info.iterrows():

    startDir=row.path+'\\'+row.folder+'\\'
    print('processing: ' + startDir)
    if not os.path.exists(startDir):
        print('WARNING: path does not exist: ' + startDir)
        continue

    
    posPath.append(glob.glob(startDir+'PositionTxt*')[0]) #this is the trajectory file
    PLPath.append(glob.glob(startDir+'PL*')[0]) # this is the pair list file
    
    head, tail = os.path.split(posPath[-1])
    currTime='dummy' #datetime.strptime(tail[-23:-4], '%Y-%m-%dT%H_%M_%S')
    expTime.append(currTime)
    
    camHeightAll.append(camHeight[('_dn_' in head)*1]) ######### this needs to be adapted for new data, since the camHeight is not stored in the meta data anymore.
    
    anNrs=row.anNr #Note that anNrs are 1 based!
    if ':' in anNrs:
        a,b=anNrs.split(sep=':')
        anNrs=np.arange(int(a),int(b)+1)
    else:
        anNrs=np.array(anNrs.split()).astype(int)
        
    anIDs=anNrs #-1 no more 0-based since using pandas merge to find animal numbers
    anIDsAll.extend(anIDs)

    bd=infoAn[infoAn.anNr.isin(anIDs)].bd.values.astype(str) # get birth dates of animals in this experiment
    #bd=infoAn.bd.values[anIDs-1] #a bit dirty to use anIDs directly here. Should merge
    birthDayAll.append(' '.join(list(bd)))

info['camHeight']=camHeightAll
info['txtPath']=posPath
info['pairList']=PLPath
info['aviPath']='default'
info['birthDayAll']=birthDayAll
info['epiDur'] = 5      # duration of individual episodes (default: 5 minutes)
info['episodes'] = 24   # number of episodes to process: -1 to load all episodes (default: -1)
info['inDish'] = 10#np.arange(len(posPath))*120     # time in dish before experiments started (default: 10)
info['arenaDiameter_mm'] = 70 # arena diameter (default: 100 mm)
info['minShift'] = 60 # minimum number of seconds to shift for control IAD
info['episodePLcode'] = 0 # flag if first two characters of episode name encode animal pair matrix (default: 0)
info['recomputeAnimalSize'] = 1 # flag to compute animals size from avi file (takes time, default: 1)
info['SaveNeighborhoodMaps'] = 0 # flag to save neighborhood maps for subsequent analysis (takes time, default: 1)
info['computeLeadership'] = 0 # flag to compute leadership index (takes time, default: 1)
info['ComputeBouts'] = 1 # flag to compute swim bout frequency (takes time, default: 1)
#info['set'] = np.arange(len(posPath))   # experiment set: can label groups of experiments (default: 0)
info['ProcessingDir']=ProcessingDir
info['outputDir']=outputDir
info['expTime']=expTime
info['readLim'] = 24*5*60*30+11

In [ ]:
csvFile=os.path.join(ProcessingDir,'processingSettings.csv')
info.to_csv(csvFile,encoding='utf-8')
info.tail()

In [ ]:
rereadData=1
if rereadData:
    def readExperiment(keepData=True):
        tmp=es.experiment_set(csvFile=csvFile,MissingOnly=True)
        if keepData:
            return tmp
        else:
            return 1

    expSet=readExperiment(keepData=False)

In [ ]:
csvPath = []
for f in [mu.splitall(x)[-1][:-4] for x in info.txtPath]:
    csvPath.append(glob.glob(ProcessingDir+f+'*siSummary*.csv')[0])

df=pd.DataFrame()
i=0
for fn in csvPath:
    print(fn)
    tmp=pd.read_csv(fn,index_col=0,sep=',')
    tmp.animalSet=i
    tmp.animalIndex=tmp.animalIndex+((i)*35)
    tmp.animalIndex=np.array(anIDsAll)[tmp.animalIndex]
    df=pd.concat([df,tmp])
    i+=1
df['episode']=[x.strip().replace('_','') for x in df['episode']]
df=pd.merge(df,infoAn[['anNr','line','genotype']],left_on='animalIndex',right_on='anNr',how='left')
df=pd.merge(df,info[['date']],left_on='animalSet',right_on=info.index,how='left')
df['setup'] = info['setup'].values[df['animalSet'].values]
print('df shape',df.shape)
df['lineSet']=[x+'_'+y for x,y in zip(df.line, df.date)]


In [ ]:
#df.tail(40)
df
#df.animalSet.unique()

In [ ]:
sns.set_palette('viridis',3)
co=sns.color_palette("viridis", 3)
idx=(df['inDishTime']<240) & (df['inDishTime']>80) # PUT THIS SETTING ON THE TOP - ANALYSIS WINDOW BETWEEN 80 AND 240s
dfDR=df[idx]
dfEpiAn=dfDR.groupby(['episode','animalIndex','line','setup','genotype','date','lineSet'],sort=True).mean(numeric_only=True).reset_index()

In [ ]:
#dfEpiAn.head()
dfEpiAn

In [ ]:

def sem(x):
    return np.std(x, ddof=1) / np.sqrt(len(x))

x=np.random.random(100)

def ci95(x):
    return np.nanmean(x)-sms.DescrStatsW(x[np.isfinite(x)]).tconfint_mean()[0]

print('std of uniform = 0.2886751345948129. STDdata:',np.std(x), 'semData:',sem(x),'samples:',x.shape) 
print('ci95:',ci95(x))

In [ ]:
dfPlot=(df.groupby(['inDishTime','episode','genotype','lineSet']).si.agg(['mean','std',sem,ci95])
    .unstack()
    .stack(dropna=True)
    .reset_index())

#dfPlot.head()
dfPlot

## Shoaling index throughout the experiment and between setups

In [ ]:
#dfPlot_filtered = dfPlot[dfPlot['genotype'] != 'esc']
dfPlot_filtered = dfPlot[~dfPlot['genotype'].str.contains(r'^(esc_(lo|hi)|na)$', regex=True)]


markers = {'hi': 'o', 'lo': 's'}  # adjust to your genotypes


g = sns.FacetGrid(
    dfPlot_filtered,
    col='lineSet',         # Facet by genotype (change as needed)
    col_wrap=3,             # Adjust for compactness
    sharex=True,
    sharey=True,
    height=2,
    aspect=1
)
g.map_dataframe(
    sns.scatterplot,
    x='inDishTime',
    y='mean',
    hue='episode',
    style='genotype',
    size=10,
    markers=markers,
    legend='full',

)
g.set(xlim=(0, 2.5*60), ylim=(0, .7))
g.set_axis_labels('Time (Minutes)', 'Attraction')
g.set_titles('{col_name}')
g.figure.subplots_adjust(top=0.85)
g.figure.suptitle('Mean attraction, all animals', fontsize=14)

# Remove the default legend and add a correct one
for ax in g.axes.flatten():
    handles, labels = ax.get_legend_handles_labels()
    ax.legend_.remove() if hasattr(ax, "legend_") and ax.legend_ else None

# Add a single legend to the figure
handles, labels = g.axes[0].get_legend_handles_labels()
g.figure.legend(handles, labels, ncol=1, handletextpad=0, bbox_to_anchor=(1, 1), loc='upper left')

# Save the plot using the path variable
g.figure.savefig(os.path.join(outputDir, "attraction_linearvsbout.pdf"), bbox_inches="tight")

outputDir

In [ ]:
dfPlot_filtered2 = dfPlot[(dfPlot['genotype'] != 'esc_hi') & (dfPlot['genotype'] != 'esc_lo') & (dfPlot['episode'] != '01k01f')]

g = sns.FacetGrid(
    dfPlot_filtered2,
    col='lineSet',         # Facet by genotype (change as needed)
    col_wrap=3,             # Adjust for compactness
    sharex=True,
    sharey=True,
    height=2,
    aspect=1
)
g.map_dataframe(
    sns.scatterplot,
    x='inDishTime',
    y='mean',
    hue='genotype',
    hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'],
    
)
#g.set(xlim=(0, 2.5*60), ylim=(0, .5))
g.set(xlim=(0, 2.5*60), ylim=(0, .7))
g.set_axis_labels('Time (Minutes)', 'Attraction')
g.set_titles('{col_name}')
g.figure.subplots_adjust(top=0.85)
g.figure.suptitle('Mean attraction, all animals', fontsize=14)

# Remove the default legend and add a correct one
for ax in g.axes.flatten():
    handles, labels = ax.get_legend_handles_labels()
    ax.legend_.remove() if hasattr(ax, "legend_") and ax.legend_ else None

# Add a single legend to the figure
handles, labels = g.axes[0].get_legend_handles_labels()
g.figure.legend(handles, labels, ncol=1, handletextpad=0, bbox_to_anchor=(1, 1), loc='upper left')

# Save the plot using the path variable
g.figure.savefig(os.path.join(outputDir, "attraction_hivslo.pdf"), bbox_inches="tight")



In [ ]:
sns.pointplot(data=dfEpiAn[dfEpiAn.episode=='02k20f'],x='lineSet',y='si',hue='setup',linestyle='none')
plt.ylim([0,.4])
plt.xticks(rotation=90);
plt.ylabel('Shoaling Index');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "shoalindex_setups.pdf"), bbox_inches="tight")



## Shoaling index across experiments

In [ ]:
fig, axes = plt.subplots(figsize=(8,3))

#All data
#ix=(dfEpiAn.episode=='02k20f')

#Only hi and lo genotypes, exclude some lineSets for experiments that had issues
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo')) & ((dfEpiAn.date!='01-07-2025') & (dfEpiAn.date!='01-09-2025') & (dfEpiAn.date!='23-04-2026'))
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo')) & ((dfEpiAn.date!='2025-07-01 00:00:00') & (dfEpiAn.date!='2025-09-01 00:00:00')) # filter out inverted experiment
 


selDat=dfEpiAn[ix]

#Settings

show_lines = "All" 
#show_lines = [ "TLSel3_7","Sel3_7", "Sel3_8"]  # set the lines you want to keep
#show_lines = [ "TLSel1_9","Sel1_10", "Sel1_11"]  # set the lines you want to keep
#show_lines = [ "Sel1_GC"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

# Define custom colors for each genotype
custom_palette = {'hi': "#D2789C", 'lo': "#25B66D"}  

sns.pointplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
             linestyle='-',
             errorbar='sd',
             dodge=.5,
             palette=custom_palette)
sns.despine()

axes.set_ylim([-0.03,0.7])
axes.set_ylabel('Shoaling Index')
axes.axhline(0,ls=':',color='k')
axes.set_title('Shoaling Index across experiments');
plt.legend(title='line',ncol=2,handletextpad=0,bbox_to_anchor=(1, 1.05))
plt.xticks(rotation=90);




In [ ]:
dfEpiAn.line.unique() #check existing lines
dfEpiAn.date.unique()

In [ ]:
fig, axes = plt.subplots(figsize=(8, 3))
#ix=(dfEpiAn.episode=='02k20f')
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo')) & (dfEpiAn.date!='01-09-2025') # filter out inverted experiment
#ix=(dfEpiAn.episode=='01k01f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo')) & (dfEpiAn.date!='01-09-2025') # filter out inverted experiment
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo')) & ((dfEpiAn.date!='2025-07-01 00:00:00') & (dfEpiAn.date!='2025-09-01 00:00:00')) # filter out inverted experiment
 


selDat=dfEpiAn[ix]

#Settings

show_lines = "All" 
#show_lines = [ "TLSel3_7","Sel3_7", "Sel3_8"]  # set the lines you want to keep
#show_lines = [ "TLSel1_9","Sel1_10", "Sel1_11"]  # set the lines you want to keep
#show_lines = [ "Sel1_GC"]  # set the lines you want to keep
xlegend_orientation = 'vertical' #choose verticar or horizontal


if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

sns.swarmplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              zorder=-1,
              dodge=.5,
              size=5,
              alpha=.5,
              #order=allCat,
             #hue_order=["hi", "lo",'wt','esc','mid'])
             hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])

sns.pointplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              dodge=.5,
              linestyle='-',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])


sns.despine()
axes.set_ylim([-0.03,0.7])

axes.set_ylabel('Attraction')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');


def split_label(label):
    label_str = str(label)

    # Split on the last underscore ( format is "LineID_Date")
    parts = label_str.rsplit('_', 1)
    return parts[0] + '\n' + parts[1]


if xlegend_orientation == 'horizontal':
    axes.set_xticklabels([split_label(t.get_text()) for t in axes.get_xticklabels()], rotation=0)
else:
    plt.xticks(rotation=90);


handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('Selection per experiment');

# Save the plot using the path variable

if show_lines == "All":
    plt.savefig(os.path.join(outputDir, "shoalindex_hivslo.pdf"), bbox_inches="tight")


## Checking delta and ratio SI between bout and linear dots

In [ ]:
ix=((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))

mainDat=dfEpiAn[ix]



#Extract ratio of shoaling index between linear and bout experiments
linear_data=mainDat[mainDat.episode=='01k01f']
bout_data=mainDat[mainDat.episode=='02k20f']

ratio_data = bout_data.copy()
ratio_data['si'] = (linear_data['si'].values/bout_data['si'].values)

ratio_data['si'].values


fig, axes = plt.subplots(figsize=(8, 3))



selDat=ratio_data

show_lines = "All" 
#show_lines = [ "Sel1_10", "Sel1_11"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

sns.swarmplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              zorder=-1,
              dodge=.5,
              size=5,
              alpha=.5,
              #order=allCat,
             #hue_order=["hi", "lo",'wt','esc','mid'])
             hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])

sns.pointplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              dodge=.5,
              linestyle='none',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])


sns.despine()

axes.set_ylabel('Ratio SI (linear/bout)')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');
#Limit y axis to better see differences
axes.set_ylim([-5,8])

plt.xticks(rotation=90);

handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('WORK IN PROGRESS - Ratio SI per experiment');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "shoalingratio_hivslo.pdf"), bbox_inches="tight")


In [ ]:
#Extract delta of shoaling index between linear and bout experiments
linear_data=dfEpiAn[dfEpiAn.episode=='01k01f']
bout_data=dfEpiAn[dfEpiAn.episode=='02k20f']

delta_data = bout_data.copy()
delta_data['si'] = bout_data['si'].values - linear_data['si'].values

delta_data['si'].values


fig, axes = plt.subplots(figsize=(8, 3))

selDat=delta_data
show_lines = "All" 
#show_lines = [ "Sel1_10", "Sel1_11"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

sns.swarmplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              zorder=-1,
              dodge=.5,
              size=5,
              alpha=.5,
              #order=allCat,
             #hue_order=["hi", "lo",'wt','esc','mid'])
             hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])

sns.pointplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              dodge=.5,
              linestyle='none',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])


sns.despine()

axes.set_ylabel('Delta SI (bout-linear)')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');

plt.xticks(rotation=90);

handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('Delta SI per experiment');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "shoalindelta_hivslo.pdf"), bbox_inches="tight")


## Checking size, speed and thigmotaxis across experiments

### Fish size across experiments

In [ ]:
fig, axes = plt.subplots(figsize=(8, 3))
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))

selDat=dfEpiAn[ix]

show_lines = "All" 
#show_lines = [ "Sel1_10", "Sel1_11"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

# sns.swarmplot(data=selDat,
#               x='lineSet',
#               y='anSize',
#               hue='genotype',
#               zorder=-1,
#               dodge=.5,
#               size=4,
#               alpha=.5,
#               #order=allCat,
#              #hue_order=["hi", "lo",'wt','esc','mid'])
#              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])

sns.pointplot(data=selDat,
              x='lineSet',
              y='anSize',
              hue='genotype',
              dodge=.5,
              linestyle='-',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])


sns.despine()

axes.set_ylabel('animal Size (mm?)')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');

plt.xticks(rotation=90);

handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('animal sizes across experiments');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "anSize_hivslo.pdf"), bbox_inches="tight")



In [ ]:

# #Subset data for only experiments happening from August 2025
# dfEpiAn.date = pd.to_datetime(dfEpiAn.date, format='%d-%m-%Y', dayfirst=True)
# dfEpiAn.date.dtype

# date_cutoff = pd.to_datetime('01-08-2025', format='%d-%m-%Y', dayfirst=True)

# dfEpiAn[dfEpiAn.date>=date_cutoff]
dfEpiAn

In [ ]:
from scipy import stats

#Correlation between SI and animal size
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))

#Subset data for only experiments happening from August 2025
subset_by_date = True # set to True to subset by date, False to keep all data
date_cutoff1 = pd.to_datetime('01-08-2025', format='%d-%m-%Y', dayfirst=True)
date_cutoff2 = pd.to_datetime('01-01-2026', format='%d-%m-%Y', dayfirst=True)

if subset_by_date == True:
    dfEpiAn.date = pd.to_datetime(dfEpiAn.date, format='%d-%m-%Y', dayfirst=True)
    pd.api.types.is_datetime64_any_dtype(dfEpiAn.date)
    ix=ix & (dfEpiAn.date<date_cutoff2)

data_for_plot = dfEpiAn[ix].groupby(['animalIndex','genotype']).mean(numeric_only=True).reset_index()

# Drop columns with missing data on the anSize column
data_for_plot = data_for_plot.dropna(subset=['anSize'])

# Define custom colors for each genotype
custom_palette = {'hi': "#D2789C", 'lo': "#25B66D"}  

# Increase font sizes for this plot
#font_scale = 1.3
#sns.set_context("notebook", font_scale=font_scale)

g=sns.FacetGrid(data_for_plot, col='genotype', hue='genotype', palette=custom_palette, height=5)

# Create a counter to track which setup we're plotting
call_count = [0]

def plot_with_regression(x, y, **kwargs):
    # Get the color from kwargs
    color = kwargs.get('color', 'C0')
    
    # Scatter plot
    plt.scatter(x, y, s=50, alpha=0.7, **kwargs)
    
    # Add regression line and calculate R²
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    r_squared = r_value**2
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = slope * x_line + intercept
    #plt.plot(x_line, y_line, **kwargs, linewidth=2)
    
    # Position labels above the plot area to avoid blocking data points
    # First setup at 1.10, second setup at 1.00
    # y_pos = 1.20 if call_count[0] % 2 == 0 else 1.10
    # call_count[0] += 1
    y_pos = 1.10
    x_pos = 0.1
    
    # Add R² text annotation outside the plot area
    plt.text(0.05, y_pos, f'R² = {r_squared:.3f}', 
             transform=plt.gca().transAxes, 
             verticalalignment='top',
             color=color,
             fontsize=12,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

g.map(plot_with_regression, 'si', 'anSize')

g.set_axis_labels('Shoaling Index', 'Animal Size', fontsize=12)
g.set_titles(col_template="{col_name}", size=12)
for ax in g.axes.flat:
    ax.tick_params(labelsize=10)

plt.legend(fontsize=10, title_fontsize=10)

data_for_plot



### Fish speed across experiments

In [ ]:
fig, axes = plt.subplots(figsize=(8, 3))
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))&(dfEpiAn.avgSpeed<10) #filter extreme speed values

selDat=dfEpiAn[ix]

show_lines = "All" 
#show_lines = [ "Sel1_10", "Sel1_11"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

# sns.swarmplot(data=selDat,
#               x='lineSet',
#               y='avgSpeed',
#               hue='genotype',
#               zorder=-1,
#               dodge=.5,
#               size=4,
#               alpha=.5,
#               #order=allCat,
#              #hue_order=["hi", "lo",'wt','esc','mid'])
#              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])

sns.pointplot(data=selDat,
              x='lineSet',
              y='avgSpeed',
              hue='genotype',
              dodge=.5,
              linestyle='-',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])


sns.despine()

axes.set_ylabel('average Speed (mm/s?)')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');

axes.set_ylim(0, 15)

plt.xticks(rotation=90);

handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('average speeds across experiments');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "avgSpeed_hivslo.pdf"), bbox_inches="tight")


In [ ]:
# Correlation between SI and average speed

from scipy import stats

#Correlation between SI and animal size
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))&(dfEpiAn.avgSpeed<10) #filter extreme speed values
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))

data_for_plot = dfEpiAn[ix].groupby(['animalIndex','genotype']).mean(numeric_only=True).reset_index()

# Drop columns with missing data on the anSize column
data_for_plot = data_for_plot.dropna(subset=['anSize'])


# Define custom colors for each genotype
custom_palette = {'hi': "#D2789C", 'lo': "#25B66D"}  

# Create a single figure with subplots (2 rows x 2 columns)
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
fig.suptitle('Correlations with AvgSpeed', fontsize=14, y=0.995)

# Define the three comparisons (thigmotaxis on x-axis, other variables on y-axis)
comparisons = [
    ('avgSpeed', 'si', 'Average Speed', 'Shoaling Index'),
    ('avgSpeed', 'anSize', 'Average Speed', 'Animal Size')
]

genotypes = ['hi', 'lo']

for row_idx, (x_var, y_var, x_label, y_label) in enumerate(comparisons):
    for col_idx, genotype in enumerate(genotypes):
        ax = axes[row_idx, col_idx]
        
        # Filter data for this genotype
        geno_data = data_for_plot[data_for_plot['genotype'] == genotype]
        color = custom_palette[genotype]
        
        # Scatter plot
        ax.scatter(geno_data[x_var], geno_data[y_var], s=50, alpha=0.7, color=color)
        
        
        # Add regression line and calculate R²
        slope, intercept, r_value, p_value, std_err = stats.linregress(geno_data[x_var], geno_data[y_var])
        r_squared = r_value**2
        x_line = np.linspace(geno_data[x_var].min(), geno_data[x_var].max(), 100)
        y_line = slope * x_line + intercept
       # ax.plot(x_line, y_line, color=color, linewidth=2)
        
        # Add R² text annotation
        ax.text(0.05, 1.10, f'R² = {r_squared:.3f}', 
                transform=ax.transAxes, 
                verticalalignment='top',
                color=color,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Set labels and title
        if row_idx == 0:
            ax.set_title(f'{genotype.upper()}', fontsize=12, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(y_label)
        if row_idx == 1:
            ax.set_xlabel(x_label)

plt.tight_layout()
plt.show()


### Thimogtaxis across experiments



In [ ]:
fig, axes = plt.subplots(figsize=(8, 3))
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))#&(~dfEpiAn.lineSet.str.contains('_2_'))

selDat=dfEpiAn[ix]

show_lines = "All" 
#show_lines = [ "Sel1_10", "Sel1_11"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

# sns.swarmplot(data=selDat,
#               x='lineSet',
#               y='thigmoIndex',
#               hue='genotype',
#               zorder=-1,
#               dodge=.5,
#               size=4,
#               alpha=.5,
#               #order=allCat,
#              #hue_order=["hi", "lo",'wt','esc','mid'])
#              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])

sns.pointplot(data=selDat,
              x='lineSet',
              y='thigmoIndex',
              hue='genotype',
              dodge=.5,
              linestyle='-',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])


sns.despine()

axes.set_ylabel('thigmotaxis Index')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');
axes.set_ylim(0, 30)

plt.xticks(rotation=90);

handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('thigmotaxis Index across experiments');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "thigmoIndex_hivslo.pdf"), bbox_inches="tight")

# Correlation between SI and all the other metrics

In [ ]:
# Correlation between SI and all the others

from scipy import stats

#Correlation between SI and animal size
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))&(dfEpiAn.boutDur<6) #filter extreme bout duration values
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))

#Filter settings. Values empirically derived from looking at the distributions, can be adjusted as needed to filter out extreme outliers that may be due to tracking errors or sick fish.
max_speed = 10
min_speed = 1 
max_boutDur = 5 
min_boutDur = 0.1 
max_thigmoIndex = 30 
min_thigmoIndex = 10 



data_for_plot = dfEpiAn[ix].groupby(['animalIndex','genotype']).mean(numeric_only=True).reset_index()

# Drop columns with missing data on the anSize or thigmoIndex column
data_for_plot = data_for_plot.dropna(subset=['anSize', 'thigmoIndex', 'boutDur'])

# Define custom colors for each genotype
custom_palette = {'hi': "#D2789C", 'lo': "#25B66D"}  

# Create a single figure with subplots, sharing both x and y within each row so each pair uses the same scale
fig, axes = plt.subplots(4, 2, figsize=(10, 16), sharex='row', sharey='row')
fig.suptitle('Correlations between Shoaling Index and Other Variables', fontsize=14, y=0.995)

# Define the three comparisons (thigmotaxis on x-axis, other variables on y-axis)
comparisons = [
    ('si', 'thigmoIndex', 'Shoaling Index', 'Thigmotaxis Index'),
    ('si', 'avgSpeed_smooth', 'Shoaling Index', 'Average Speed'),
    ('si', 'anSize', 'Shoaling Index', 'Animal Size'),
    ('si', 'boutDur', 'Shoaling Index', 'Bout Duration')
]

# comparisons = [
#     ('si', 'thigmoIndex', 'Shoaling Index', 'Thigmotaxis Index'),
#     ('si', 'avgSpeed_smooth', 'Shoaling Index', 'Average Speed')
#     # ('si', 'anSize', 'Shoaling Index', 'Animal Size'),
#     # ('si', 'boutDur', 'Shoaling Index', 'Bout Duration')
# ]

genotypes = ['hi', 'lo']

for row_idx, (x_var, y_var, x_label, y_label) in enumerate(comparisons):
    for col_idx, genotype in enumerate(genotypes):
        ax = axes[row_idx, col_idx]
        
        # Filter data for this genotype
        geno_data = data_for_plot[data_for_plot['genotype'] == genotype]
        color = custom_palette[genotype]
        
        # Scatter plot
        ax.scatter(geno_data[x_var], geno_data[y_var], s=50, alpha=0.7, color=color)
        
        # Threshold guides for the variables used in the filtering step
        if y_var == 'thigmoIndex':
            ax.axhline(min_thigmoIndex, ls='--', color='k', alpha=0.25, linewidth=1)
            ax.axhline(max_thigmoIndex, ls='--', color='k', alpha=0.25, linewidth=1)
        elif y_var == 'avgSpeed_smooth':
            ax.axhline(min_speed, ls='--', color='k', alpha=0.25, linewidth=1)
            ax.axhline(max_speed, ls='--', color='k', alpha=0.25, linewidth=1)
        elif y_var == 'boutDur':
            ax.axhline(min_boutDur, ls='--', color='k', alpha=0.25, linewidth=1)
            ax.axhline(max_boutDur, ls='--', color='k', alpha=0.25, linewidth=1)
        
        # Add regression line and calculate R²
        slope, intercept, r_value, p_value, std_err = stats.linregress(geno_data[x_var], geno_data[y_var])
        r_squared = r_value**2
        x_line = np.linspace(geno_data[x_var].min(), geno_data[x_var].max(), 100)
        y_line = slope * x_line + intercept
        #ax.plot(x_line, y_line, color=color, linewidth=2)
        
        # Add R² text annotation
        ax.text(0.05, 1.10, f'R² = {r_squared:.3f}', 
                transform=ax.transAxes, 
                verticalalignment='top',
                color=color,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Set labels and title
        if row_idx == 0:
            ax.set_title(f'{genotype.upper()}', fontsize=12, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(y_label)
        if row_idx == len(comparisons) - 1:
            ax.set_xlabel(x_label)

plt.tight_layout()
plt.show()

data_for_plot

# Re-analyze shoaling data but filter extreme values of boutDur, avgspeed and thigmotaxis

In [ ]:
fig, axes = plt.subplots(figsize=(8, 3))
#ix=(dfEpiAn.episode=='02k20f')
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo')) & (dfEpiAn.date!='01-09-2025')  # filter out inverted experiment
#ix=(dfEpiAn.episode=='01k01f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo')) & (dfEpiAn.date!='01-09-2025') # plot data for linear stimuli
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo')) & ((dfEpiAn.date!='2025-07-01 00:00:00') & (dfEpiAn.date!='2025-09-01 00:00:00')) # filter out inverted experiment
 


preselDat=dfEpiAn[ix]

#Filter data based on the settings defined in the correlation plot to exclude extreme outliers that may be due to tracking errors or sick fish. The thresholds can be adjusted as needed.
selDat = preselDat[(preselDat.avgSpeed<max_speed) & (preselDat.avgSpeed>min_speed) & (preselDat.boutDur<max_boutDur) & (preselDat.boutDur>min_boutDur) & (preselDat.thigmoIndex<max_thigmoIndex) & (preselDat.thigmoIndex>min_thigmoIndex)]
unselDat = preselDat[(preselDat.avgSpeed>max_speed) | (preselDat.avgSpeed<min_speed) | (preselDat.boutDur>max_boutDur) | (preselDat.boutDur<min_boutDur) | (preselDat.thigmoIndex>max_thigmoIndex) | (preselDat.thigmoIndex<min_thigmoIndex)]

#Settings

show_lines = "All" 
#show_lines = [ "TLSel3_7","Sel3_7", "Sel3_8"]  # set the lines you want to keep
#show_lines = [ "TLSel1_9","Sel1_10", "Sel1_11"]  # set the lines you want to keep
#show_lines = [ "Sel1_GC"]  # set the lines you want to keep
xlegend_orientation = 'vertical' #choose verticar or horizontal


if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]



sns.pointplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              dodge=.5,
              linestyle='-',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])

sns.swarmplot(data=unselDat,
              x='lineSet',
              y='si',
              hue='genotype',
              zorder=-1,
              dodge=.5,
              size=5,
              alpha=.5,
              #order=allCat,
             #hue_order=["hi", "lo",'wt','esc','mid'])
             hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])


sns.despine()
axes.set_ylim([-0.03,0.7])

axes.set_ylabel('Attraction')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');


def split_label(label):
    label_str = str(label)

    # Split on the last underscore ( format is "LineID_Date")
    parts = label_str.rsplit('_', 1)
    return parts[0] + '\n' + parts[1]


if xlegend_orientation == 'horizontal':
    axes.set_xticklabels([split_label(t.get_text()) for t in axes.get_xticklabels()], rotation=0)
else:
    plt.xticks(rotation=90);


handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('Selection per experiment');

# Save the plot using the path variable

if show_lines == "All":
    plt.savefig(os.path.join(outputDir, "shoalindex_hivslo.pdf"), bbox_inches="tight")


# Generate Grid overview which animals to select

select one experimental condition to be analyzed and plotted based on lineSet as 'ShowGroup' below.
This will typically be from two setups.

In [ ]:
dfEpiAn.lineSet.unique()

In [ ]:
ShowGroup='Sel1_GC8_07-05-2026' # this is the group we want to select fish from
nSel=10 #how many fish to select, typically 10

In [ ]:
ix=(dfEpiAn.episode=='02k20f')&(dfEpiAn.lineSet==ShowGroup)

selDat=dfEpiAn[ix].copy()
selDat['plotGroup']=0

selDat['rank']=selDat.groupby(['lineSet','genotype'])['si'].rank()
selDat['rankInverse']=selDat.groupby(['lineSet','genotype'])['si'].rank(ascending=False)
selDat['pick']=((((selDat['rank']<=nSel)&(selDat['genotype']=='lo')) | ((selDat.rankInverse<=nSel)&(selDat['genotype']=='hi'))))


sns.swarmplot(selDat,x='genotype',y='si',color='gray');
sns.swarmplot(selDat[selDat['pick']==1 ],x='genotype',y='si',hue='genotype');
plt.title(ShowGroup + ' selection of ' + str(nSel) + ' fish per group');
plt.ylabel('Attraction');
plt.xlabel('Group');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "selectfish_dots.pdf"), bbox_inches="tight")




In [ ]:
#dfEpiAn.head()
selDat.head()

In [ ]:
#thigmotaxis x si

ix=(dfEpiAn.episode=='02k20f')&(dfEpiAn.lineSet==ShowGroup)
g=sns.FacetGrid(dfEpiAn[ix].groupby(['setup','animalID','genotype']).mean(numeric_only=True).reset_index(),col='genotype',hue='setup')
#g.set(xlim=(0,.5))

g=g.map(plt.scatter,'si','thigmoIndex',s=50,alpha=0.7)
plt.legend()


In [ ]:
#avgspeed x si

ix=(dfEpiAn.episode=='02k20f')&(dfEpiAn.lineSet==ShowGroup)
g=sns.FacetGrid(dfEpiAn[ix].groupby(['setup','animalID','genotype']).mean(numeric_only=True).reset_index(),col='genotype',hue='setup')
#g.set(xlim=(0,.5))

g=g.map(plt.scatter,'si','avgSpeed',s=50,alpha=0.7)
plt.legend()


In [ ]:
ix=(dfEpiAn.episode=='02k20f')&(dfEpiAn.lineSet==ShowGroup)
g=sns.FacetGrid(dfEpiAn[ix].groupby(['setup','animalID','genotype']).mean(numeric_only=True).reset_index(),col='genotype',hue='setup')
#g.set(xlim=(0,.5))

g=g.map(plt.scatter,'si','anSize',s=50,alpha=0.7)
plt.legend()

In [ ]:
ixlo=(dfEpiAn.episode=='02k20f')&(dfEpiAn.genotype=='lo')&(dfEpiAn.lineSet==ShowGroup)
dataLo=dfEpiAn[ixlo].groupby(['setup','animalID','genotype','avgSpeed','thigmoIndex']).si.mean().reset_index()
dataLo['rank']=dataLo.si.rank()
dataLo=dataLo.sort_values(by='rank').reset_index()
dataLo

In [ ]:

ixhi=(dfEpiAn.episode=='02k20f')&(dfEpiAn.genotype=='hi')&(dfEpiAn.lineSet==ShowGroup)
dataHi=dfEpiAn[ixhi].groupby(['setup','animalID','genotype','avgSpeed','thigmoIndex']).si.mean(numeric_only=True).reset_index()
dataHi['rank']=dataHi.si.rank()
dataHi=dataHi.sort_values(by='rank',ascending =False).reset_index()
dataHi

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(3.5, 5))


major_ticks = np.arange(0, 7, 7)
minor_ticks = np.arange(0, 7, 1)



for n,ax in enumerate(axes):
    for i in range(35):
        ax.text(i%7+.5,4-np.floor_divide(i,7)+.5,str(i+1))
        
    
    ax.set_xticks(major_ticks)
    ax.set_xticks(minor_ticks, minor=True)
    ax.set_yticks(major_ticks)
    ax.set_yticks(minor_ticks, minor=True)

    ax.set_xlim([0,7])
    ax.set_ylim([0,5])
    ax.grid(which='both')
    ax.tick_params(labelbottom=False)    
    ax.tick_params(labelleft=False)  
    
for n,row in dataHi[:nSel].iterrows():
    a=int(row.setup==2)
    i=row.animalID
    axes[a].plot(i%7+.5,4-np.floor_divide(i,7)+.5,'r.')
    axes[a].text(i%7+.2,4-np.floor_divide(i,7)+.1,'hi:'+str(n),color='r',fontsize=8)

for n,row in dataLo[:nSel].iterrows():
    a=int(row.setup==2)
    i=row.animalID
    axes[a].plot(i%7+.5,4-np.floor_divide(i,7)+.5,'b.')
    axes[a].text(i%7+.2,4-np.floor_divide(i,7)+.1,'lo:'+str(n),color='b',fontsize=8)
    

#axes[0].set_title('setup1 '+ShowGroup + ' ' +dfEpiAn[ixhi].date.unique()[0]);
#axes[1].set_title('setup2 '+ShowGroup + ' ' +dfEpiAn[ixhi].date.unique()[0]);

axes[0].set_title('setup1 '+ShowGroup);
axes[1].set_title('setup2 '+ShowGroup);

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "selectfish_grid.pdf"), bbox_inches="tight")


In [ ]:
ix=(dfEpiAn.episode=='02k20f')
g=sns.FacetGrid(dfEpiAn[ix].groupby(['setup','animalID','genotype','lineSet']).mean(numeric_only=True).reset_index(),col='genotype')
g=g.map(plt.scatter,'si','boutDur',s=10,alpha=0.5)
for ax in g.axes.ravel():
    ax.set_ylim([0.2,1.5])
    ax.axhline(20/30)
#figPath=base+'SelectionSizeVsShoal.png'
#plt.savefig(figPath,bbox_inches='tight')


In [ ]:
ix=(dfEpiAn.episode=='01k01f')
g=sns.FacetGrid(dfEpiAn[ix].groupby(['setup','animalID','genotype','lineSet']).mean(numeric_only=True).reset_index(),col='genotype')
g=g.map(plt.scatter,'si','boutDur',s=10,alpha=0.5)
for ax in g.axes.ravel():
    ax.set_ylim([0.2,1.5])
    ax.axhline(20/30)